# Llama-3.2-1B × OpenMathInstruct-2 leaderboard — robustness (9000 steps)

Cross-**model** *and* cross-**dataset** robustness in one cell: does `adam-polar-product-lora-coupled-spectral-chord-tight` (ns=8, full Newton–Schulz whitening, picard=1) keep its eval-loss edge over AdamW-LoRA when the base is **Llama-3.2-1B** (not OLMo) *and* the task is **math-IFT** (OpenMathInstruct-2, not code)?

Setup: Llama-3.2-1B × OpenMathInstruct-2 `train_2M` (Llama-tokenized cache `data/openmath_instruct_2_2m_packed_seq2048_llama32`, 512,440 packed slots @ seq=2048) × global_batch=16 (4×4) × packed_v1.1 × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell. `max_steps=9000`, `eval_every=250`.

- **AdamW**: η ∈ {3e-5, 1e-4, 3e-4, 1e-3}
- **chord-tight ns=8** (`adam-polar-product-lora-coupled-spectral-chord-tight`, `--muon_ns_steps 8 --polar_method ns`, picard=1): η ∈ {3e-3, 1e-2, 3e-2, 1e-1} — grid pre-extended to 1e-1 because OLMo×math's ns=8 pinned at the 1e-2 grid edge.

Source log groups: `adamw_robustness_llama32_1b_openmath_r{64,256}_blackwell` and `chord_tight_robustness_llama32_1b_openmath_r{64,256}_ns8_blackwell`. No ns=5 arm here (OLMo×math established ns=8 ≈ ns=5; this notebook goes straight to ns=8). Grids are single-sweep — no separate `_ext_right` groups.

**σ anchor**: no per-(model,dataset) multi-seed AdamW run. Δ quoted against `σ_AdamW(packed_v1, opc-sft-stage2, r=64) = 0.0017` as a **loose proxy only** — now cross-model *and* cross-dataset, so treat σ-units as indicative, not rigorous, until re-anchored.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.loader import load_runs
from lora_playground.plotting import compare_variants_figure, canonical_label
from IPython.display import display

# Single source of truth: canonical_label (ns/picard/damping-explicit, identical
# across every notebook), canonical colors (AdamW black + first), guard hard-errors
# on any silent merge.

def _key(cfg):
    return (cfg['optimizer'], float(cfg['lr']), cfg.get('muon_ns_steps'),
            cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')))

def render_cell(groups, rank, suptitle, *, final_ylim=None, traj_ylim=None, figsize=(11, 4)):
    runs = load_runs(where={'log_group': groups}, logs_root='../logs',
                     warn_cross_commit=False, quiet=True)
    dedup = {}
    for cfg, hist in runs:
        if cfg.get('lora_r') != rank:
            continue
        k = _key(cfg)
        if k not in dedup or len(hist) > len(dedup[k][1]):
            dedup[k] = (cfg, hist)
    labeled = [(c, h) for c, h in dedup.values() if canonical_label(c) is not None]
    labels = {canonical_label(c) for c, _ in labeled}
    fig, tdf, sdf = compare_variants_figure(
        variants={l: {} for l in labels}, common_where={}, ref_label='AdamW',
        target_label='AdamW', sigma_ref=0.0017, suptitle=suptitle, figsize=figsize,
        max_steps=9000, allow_partial=True, final_ylim=final_ylim, traj_ylim=traj_ylim,
        prefetched_runs=labeled, variant_key=canonical_label)
    plt.show()
    print(f'--- {suptitle} per-η table ---')
    display(tdf.style.format('{:.4f}', na_rep='—'))
    print(f'--- {suptitle} summary ---')
    display(sdf.style.format({'final': '{:.4f}', 'delta': '{:+.4f}',
                              'delta_sigma': '{:+.2f}σ', 'best_lr': '{:.0e}'}, na_rep='—'))
    return tdf, sdf

## r=64

In [ ]:
GROUPS_R64 = [
    'adamw_robustness_llama32_1b_openmath_r64_blackwell',
    'chord_tight_robustness_llama32_1b_openmath_r64_ns8_blackwell',
    'chord_tight_robustness_llama32_1b_openmath_r64_ns8_ext_left_blackwell',  # down-ext {1e-3, 3e-4}: ns=8 best pinned at grid-min 3e-3
]
# ylim left auto until data lands; tune after first full evals (see OLMo nb caps)
render_cell(GROUPS_R64, 64, 'Llama-3.2-1B × OpenMathInstruct-2 r=64')

## r=256

In [ ]:
GROUPS_R256 = [
    'adamw_robustness_llama32_1b_openmath_r256_blackwell',
    'chord_tight_robustness_llama32_1b_openmath_r256_ns8_blackwell',
    'chord_tight_robustness_llama32_1b_openmath_r256_ns8_ext_left_blackwell',  # down-ext {1e-3, 3e-4}: ns=8 best pinned at grid-min 3e-3
]
render_cell(GROUPS_R256, 256, 'Llama-3.2-1B × OpenMathInstruct-2 r=256')